In [1]:
import subprocess
subprocess.run(["pip","install","cdsapi","xarray","netCDF4","pandas","matplotlib"], check=True)
print("Done!")

Done!


## Configuration

In [2]:
import os
lat           = 34.499984
lon           = -4.708586
site_name     = "test_site"
startDate     = "2005-01-01"
endDate       = "2024-12-31"
selected_year = 2015
print("lat:", lat, "lon:", lon)
print("Period:", startDate, "to", endDate)

lat: 34.499984 lon: -4.708586
Period: 2005-01-01 to 2024-12-31


## Step 1 — Download ERA5 year by year
Downloads one year at a time to avoid Copernicus size limits. Each file is saved so if it crashes you can restart without re-downloading.

In [4]:
import cdsapi

start_year = int(startDate[:4])
end_year   = int(endDate[:4])
months     = [str(m).zfill(2) for m in range(1, 13)]
days       = [str(d).zfill(2) for d in range(1, 32)]
area       = [lat + 0.25, lon - 0.25, lat - 0.25, lon + 0.25]

c = cdsapi.Client()

downloaded_files = []

for year in range(start_year, end_year + 1):
    filename = site_name + "_era5_" + str(year) + ".nc"
    downloaded_files.append(filename)

    if os.path.exists(filename):
        print("Already exists, skipping:", filename)
        continue

    print("Downloading year:", year, "...")
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable":     "2m_temperature",
            "year":         str(year),
            "month":        months,
            "day":          days,
            "time":         ["00:00", "06:00", "12:00", "18:00"],
            "area":         area,
            "data_format":  "netcdf",
        },
        filename
    )
    print("Done:", filename)

print("All years downloaded!")
print("Files:", downloaded_files)

2026-05-31 22:33:08,973 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-05-31 22:33:08,975 INFO Request ID is 870b4b61-2af9-4dd2-bbce-5db7bb89514f
2026-05-31 22:33:09,105 INFO status has been updated to accepted
2026-05-31 22:33:31,523 INFO status has been updated to running
2026-05-31 22:36:04,414 INFO status has been updated to successful


142f9257294ac1b68b1f7c9669b2dfc5.nc:   0%|          | 0.00/137k [00:00<?, ?B/s]

Done: test_site_era5_2005.nc


2026-05-31 22:36:12,521 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-05-31 22:36:12,524 INFO Request ID is af3d374c-4054-4589-8f76-a4d75d46b946
2026-05-31 22:36:12,667 INFO status has been updated to accepted
2026-05-31 22:36:26,467 INFO status has been updated to running
2026-05-31 22:39:07,091 INFO status has been updated to successful


7e83d4b6747666cea0a12eaafd373d2b.nc:   0%|          | 0.00/137k [00:00<?, ?B/s]

...

2026-05-31 22:39:10,755 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-05-31 22:39:10,757 INFO Request ID is c022d503-dcc9-4dc9-b923-b831bc213068
2026-05-31 22:39:10,883 INFO status has been updated to accepted
2026-05-31 22:39:19,527 INFO status has been updated to running
2026-05-31 22:42:05,731 INFO status has been updated to successful


f261f8ef59c383e334f16a28f82ba6db.nc:   0%|          | 0.00/137k [00:00<?, ?B/s]

Done: test_site_era5_2007.nc


2026-05-31 22:42:09,813 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-05-31 22:42:09,818 INFO Request ID is 0d515e8d-51cd-4fe3-a74e-911b5aa9fc2f
2026-05-31 22:42:11,110 INFO status has been updated to accepted
2026-05-31 22:42:21,573 INFO status has been updated to running


KeyboardInterrupt: 

## Step 2 — Merge all years into one DataFrame

In [ ]:
import xarray as xr
import pandas as pd

all_dfs = []

for filename in downloaded_files:
    if not os.path.exists(filename):
        print("Missing file:", filename)
        continue

    ds   = xr.open_dataset(filename)
    temp = ds["t2m"]
    tp   = temp.sel(latitude=lat, longitude=lon, method="nearest")
    df   = tp.to_dataframe(name="tk").reset_index()
    df["temperature"] = df["tk"] - 273.15
    tcol = "valid_time" if "valid_time" in df.columns else "time"
    df["date"] = pd.to_datetime(df[tcol])
    df = df[["date", "temperature"]].copy()
    all_dfs.append(df)
    ds.close()

combined = pd.concat(all_dfs, ignore_index=True)
combined = combined[(combined["date"] >= startDate) & (combined["date"] <= endDate)]
combined["date"] = combined["date"].dt.date
daily = combined.groupby("date")["temperature"].mean().reset_index()
daily.columns = ["date", "temperature"]
daily["date"] = pd.to_datetime(daily["date"])

print("Total days:", len(daily))
print("Min temp:", round(daily["temperature"].min(), 2), "C")
print("Max temp:", round(daily["temperature"].max(), 2), "C")
daily.head(10)

## Step 3 — Analysis: best, worst, selected year

In [ ]:
daily["year"] = daily["date"].dt.year
yearly = daily.groupby("year")["temperature"].mean()

best_year  = int(yearly.idxmax())
worst_year = int(yearly.idxmin())

print("=" * 45)
print("Best year  (hottest):", best_year, "avg", round(yearly[best_year], 2), "C")
print("Worst year (coldest):", worst_year, "avg", round(yearly[worst_year], 2), "C")

if selected_year in yearly.index:
    sel = daily[daily["year"] == selected_year]
    print("
Selected year:", selected_year)
    print("  Average    :", round(yearly[selected_year], 2), "C")
    print("  Hottest day:", round(sel["temperature"].max(), 2), "C")
    print("  Coldest day:", round(sel["temperature"].min(), 2), "C")

print("
All years:")
for yr, avg in yearly.items():
    tag = ""
    if yr == best_year:     tag += " <- BEST"
    if yr == worst_year:    tag += " <- WORST"
    if yr == selected_year: tag += " <- SELECTED"
    print(" ", yr, ":", round(avg, 2), "C" + tag)
print("=" * 45)

## Step 4 — Save to CSV

In [ ]:
csv_file = site_name + "_temperature_data.csv"
daily.to_csv(csv_file, index=False)
print("Saved", len(daily), "rows to", csv_file)
daily.head()

## Step 5 — Plot

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

ax1 = axes[0]
ax1.plot(daily["date"], daily["temperature"], linewidth=0.6, color="steelblue")
ax1.set_title("Daily Temperature - " + site_name + " (lat=" + str(lat) + ", lon=" + str(lon) + ")", fontsize=14)
ax1.set_ylabel("Temperature (C)")
ax1.grid(True, alpha=0.3)
for yr, color, label in [(best_year, "red", "Best"), (worst_year, "blue", "Worst")]:
    yd = daily[daily["date"].dt.year == yr]
    ax1.axvspan(yd["date"].min(), yd["date"].max(), alpha=0.15, color=color, label=label + " " + str(yr))
ax1.legend()

ax2 = axes[1]
colors = []
for yr in yearly.index:
    if yr == best_year:       colors.append("red")
    elif yr == worst_year:    colors.append("blue")
    elif yr == selected_year: colors.append("orange")
    else:                     colors.append("steelblue")
ax2.bar(yearly.index, yearly.values, color=colors, alpha=0.8)
ax2.set_title("Yearly Average Temperature", fontsize=14)
ax2.set_xlabel("Year")
ax2.set_ylabel("Avg Temp (C)")
ax2.grid(True, alpha=0.3, axis="y")
ax2.legend(handles=[
    Patch(facecolor="red",       label="Best (" + str(best_year) + ")"),
    Patch(facecolor="blue",      label="Worst (" + str(worst_year) + ")"),
    Patch(facecolor="orange",    label="Selected (" + str(selected_year) + ")"),
    Patch(facecolor="steelblue", label="Other years"),
])
plt.tight_layout()
plt.savefig(site_name + "_plot.png", dpi=150)
plt.show()
print("Plot saved!")